# Notebook 03 - Data Preparation, Policy Mapping, and Long-Text Handling

This notebook explains every transformation from raw legal rows to instruction-response supervision.

## What Is This Technique? - Deterministic Policy Mapping for Legal Supervision

### Definition and Core Concepts
Policy mapping converts existing legal labels and text cues into the project output schema without synthetic LLM-generated labels.

### Why Was This Technique Developed?
LexGLUE labels are task-specific; we need one unified schema for summarization, risk, liability, and clause extraction.

### What Limitations of Traditional RAG Does It Solve?
RAG retrieves context but does not produce structured supervision targets for training. Policy mapping closes that training-data gap.

### Architecture and Workflow Diagram Explanation
```mermaid
flowchart LR
A[Raw legal text + gold labels] --> B[Normalization]
B --> C[Risk mapping]
B --> D[Liability mapping]
B --> E[Obligation/right/restriction extraction]
C --> F[Structured LegalAnalysis target]
D --> F
E --> F
```

### Component-by-Component Breakdown
1. Text cleaning and truncation.
2. Keyword-driven risk and liability mapping.
3. Clause cue extraction for obligations/rights/restrictions.
4. Evidence span capture for hallucination auditing.
5. Instruction + input + JSON output formatting.

### When Should It Be Used in Real-World Systems?
Use when you have real domain labels but not a ready-made instruction-tuning target format.

### Advantages and Disadvantages
Advantages:
- Auditable and deterministic.
- No synthetic labeling dependence.
- Easy to maintain and debug.

Disadvantages:
- Can miss nuanced semantics.
- Rule updates needed for new jurisdictions.
- Less expressive than expert-annotated legal analyses.

### Comparison Against Standard RAG and Other Implemented Variants
Compared with standard RAG pipelines, this approach focuses on train-time schema construction rather than inference-time retrieval.

### Implementation Details and Design Decisions Used in This Project
Policy rules live in `configs/policy_mapping.yaml`; transformations are implemented in `data/policy_mapping.py` and `data/instruction_builder.py`.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
ROOT = next((p for p in candidates if (p / 'scripts').exists() and (p / 'configs').exists()), cwd)
print('Project root:', ROOT)

def run_py(script: str, *args: str) -> None:
    cmd = [sys.executable, str(ROOT / script), *args]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=str(ROOT))


In [ ]:
train_path = ROOT / 'data/processed/train_supervised.jsonl'
if not train_path.exists():
    run_py('scripts/prepare_data.py', '--config', 'configs/default.yaml')
preview = []
with train_path.open() as f:
    for i, line in enumerate(f):
        if i >= 2:
            break
        preview.append(json.loads(line))
preview

In [ ]:
policy = json.loads((ROOT / 'data/processed/dataset_report.json').read_text())
pd.DataFrame([
    {'split': k, 'rows': v} for k, v in policy['row_counts'].items()
])

## Long-Document Handling Strategy

Current approach uses conservative truncation for local training constraints. For production, add chunking + sliding-window map-reduce summarization for very long contracts.